In [ ]:
from embedder import Embedder
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from minsearch import VectorSearch, Index

2026-06-28 20:14:43.144366921 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [ ]:
embed = Embedder()

query = "How does approximate nearest neighbor search work?"
v = embed.encode(query)
v[0]

np.float64(-0.02058203437252893)

In [5]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [ ]:
documents

In [12]:
doc = documents[22]
text = doc['filename'] + "\n\n" + doc['content']
X = embed.encode(text)

X

array([-3.19288701e-03,  6.51635799e-03, -3.07340064e-02,  2.78382188e-02,
        5.06408712e-02, -3.62153008e-03,  4.13947421e-02, -5.16299211e-03,
       -6.82227626e-02,  4.62040597e-03, -8.82862532e-03,  5.83804941e-02,
        8.24440273e-04, -4.71377750e-02, -1.24324781e-01,  2.11814364e-02,
       -6.14079401e-02,  5.77649022e-02,  9.12647571e-02,  4.09882366e-02,
       -2.78161597e-02,  3.27228350e-02,  2.92351671e-02, -1.29427478e-02,
       -1.10719006e-02,  7.99442389e-02,  1.67280247e-02, -6.04075319e-02,
        2.96045640e-02, -1.74245015e-02,  4.83352123e-02,  5.17016376e-02,
        1.41671852e-02,  1.94770771e-01, -6.89440975e-02,  5.81340853e-02,
       -6.51644753e-02, -1.13913039e-02, -7.56435147e-02, -9.76147707e-05,
       -9.97005048e-03,  3.14384648e-02, -3.44072337e-02,  5.33694778e-02,
        3.99539693e-02,  4.28576214e-02, -4.30360029e-02, -6.53757086e-02,
        7.63227005e-02, -7.34869951e-02, -1.14360828e-01, -2.88512158e-02,
       -3.14937885e-02, -

In [14]:
print("Cosine Similarity - ", v.dot(X))

Cosine Similarity -  0.3750900375217231


In [17]:
chunks = chunk_documents(documents, size=2000, step=1000)

contents = [chunk['content'] for chunk in chunks]
X = embed.encode_batch(contents) 

scores = X.dot(v)

#Get the top results
top_indices = scores.argsort()[::-1]

# Print top 3 results
for i in top_indices[:3]:
    print(f"Score: {scores[i]:.4f}")
    print(f"File: {chunks[i]['filename']}")
    print(f"Content: {chunks[i]['content'][:200]}...")
    print()

Score: 0.6489
File: 02-vector-search/lessons/07-sqlitesearch-vector.md
Content: rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search take...

Score: 0.5510
File: 01-agentic-rag/lessons/05-search.md
Content: # Search

Video: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

## Search basics

At its core, every search engine does the same thing. It ta...

Score: 0.4066
File: 04-evaluation/lessons/05-search-metrics.md
Content: k mean the same thing.

Let's calculate it:

```python
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt
```

There are 14 hits. The example has 15 queries.

The Hit Rate is:
...



In [20]:
vs_index = VectorSearch(keyword_fields=["filename"])
vs_index.fit(X, chunks)


query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)


results = vs_index.search(query_vector, num_results=5)


for r in results:
    print(f"File: {r['filename']}")
    print(f"Content: {r['content'][:300]}...")
    print()

File: 04-evaluation/lessons/05-search-metrics.md
Content: # Search Evaluation Metrics

Video: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we computed relevance lists for search results.
We can turn those lists into metrics.

## Hit Rate

Hit Rate (also called Recall@k) me...

File: 04-evaluation/lessons/01-intro.md
Content: # Evaluation

Video: [Watch this lesson](https://www.youtube.com/watch?v=eC_IcxfxoiQ&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous modules, we built search engines and RAG pipelines.
We tried different approaches: keyword search with minsearch, vector
search, agents with function calling...

File: 01-agentic-rag/lessons/05-search.md
Content: # Search

Video: [Watch this lesson](https://www.youtube.com/watch?v=GYgpNKiuCJU&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

## Search basics

At its core, every search engine does the same thing. It takes a query,
scores every docume

In [22]:
kw_index = Index(text_fields=["content"], keyword_fields=["filename"])
kw_index.fit(chunks)


query = "How do I store vectors in PostgreSQL?"
query_vector = embed.encode(query)


kw_results = kw_index.search(query, num_results=5)
vs_results = vs_index.search(query_vector, num_results=5)

# --- Compare ---
kw_files = set(r['filename'] for r in kw_results)
vs_files = set(r['filename'] for r in vs_results)

print("Keyword results:")
for r in kw_results:
    print(f"  {r['filename']}")

print("\nVector results:")
for r in vs_results:
    print(f"  {r['filename']}")

print("\nIn vector but NOT in keyword results:")
for f in vs_files - kw_files:
    print(f"  {f}")

Keyword results:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

Vector results:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md

In vector but NOT in keyword results:
  02-vector-search/lessons/08-pgvector.md


In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]



query = "How do I give the model access to tools?"
query_vector = embed.encode(query)


vector_results = vs_index.search(query_vector, num_results=5)
text_results = kw_index.search(query, num_results=5)


results = rrf([vector_results, text_results])


for r in results:
    print(f"File: {r['filename']}")
    print(f"Content: {r['content'][:300]}...")
    print()

File: 01-agentic-rag/lessons/13-function-calling.md
Content:  function. `parameters` is a JSON schema
for the arguments, and we mark `query` as required so the model always
fills it in.

## Sending the question with the tool

Now we send the same question as before, but this time we include the
tool in the request:

```python
response = openai_client.response...

File: 01-agentic-rag/lessons/01-intro.md
Content: wrong.

## The project

RAG solves these problems by giving the LLM relevant documents at
question time. We don't hope the model memorized the answer. We
retrieve the right information and hand it to the LLM, and the model
generates a grounded response. This lets us inject knowledge the model
never ...

File: 01-agentic-rag/lessons/14-agentic-loop.md
Content: # The Agentic Loop

Video: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous lesson, we did function calling by hand. We sent a
message and got back a